# Exercise 4 — Reusing Pretrained Models

**CAS Deep Learning — Computer Vision**

## Learning objectives
After this exercise you should be able to:
- Extract feature embeddings from a pretrained model
- Apply three adaptation strategies: k-NN, linear probe, and fine-tuning
- Compare accuracy vs. compute trade-offs across strategies
- Select the right strategy based on dataset size and constraints

**Fill in code where you see `# YOUR CODE HERE`.**

## Setup Colab / Local Paths

The following setup sets the default data path: Make sure to check and adapt `DATA_PATH`.

If working in Colab: **Save the notebook to your personal Google Drive to persist changes**.

Packages not in the Colab environment will be installed as well.

In [ ]:
import sys
from pathlib import Path

from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"

IN_COLAB = "google.colab" in sys.modules
print(f"In Colab: {IN_COLAB}")


if IN_COLAB:
    print("Installing additional packages to Colab Environment")
    !pip install -q -r requirements.txt

    print("Mounting Drive")
    from google.colab import drive

    ROOT_PATH = Path("/content/drive")
    drive.mount(str(ROOT_PATH))
    DATA_PATH = ROOT_PATH.joinpath("MyDrive/cas-dl-compvis")
else:
    import pyrootutils

    ROOT_PATH = pyrootutils.setup_root(
        search_from=".",
        indicator="pyproject.toml",
        project_root_env_var=True,
        dotenv=True,
        pythonpath=True,
        cwd=True,
    )

    DATA_PATH = ROOT_PATH.joinpath("data")

print(f"Creating {DATA_PATH} if it does not exist")
DATA_PATH.mkdir(parents=True, exist_ok=True)

## Imports

In [ ]:
import time

import matplotlib.pyplot as plt
import timm
import torch
import torch.nn as nn
from sklearn.metrics import classification_report
from sklearn.neighbors import KNeighborsClassifier
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from tqdm.notebook import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device} | Colab: {IN_COLAB}")

## Load the Dataset

In [ ]:
def ensure_dataset(
    data_path,
    archive_name,
    extracted_subdir,
    *,
    gdrive_id=None,
    hf_repo=None,
):
    """Check-then-download a dataset archive. Returns the extracted directory.

    Tries Google Drive first (gdown), falls back to Hugging Face Hub. Supports
    .tar.gz and .zip. Idempotent — skips if the extracted directory exists.
    """
    data_path = Path(data_path)
    dataset_dir = data_path / extracted_subdir
    if dataset_dir.exists():
        print(f"Dataset already present: {dataset_dir}")
        return dataset_dir

    data_path.mkdir(parents=True, exist_ok=True)
    archive_path = data_path / archive_name

    if not archive_path.exists():
        errors = []
        if gdrive_id is not None:
            try:
                import gdown

                url = f"https://drive.google.com/uc?id={gdrive_id}"
                print(f"Downloading {archive_name} from Google Drive ...")
                gdown.download(url, str(archive_path), quiet=False, fuzzy=True)
                if not archive_path.exists():
                    raise RuntimeError("gdown reported success but file is missing")
            except Exception as exc:
                errors.append(f"gdown: {exc}")
                archive_path.unlink(missing_ok=True)

        if not archive_path.exists() and hf_repo is not None:
            try:
                from huggingface_hub import hf_hub_download

                print(f"Downloading {archive_name} from Hugging Face ({hf_repo}) ...")
                cached = hf_hub_download(hf_repo, archive_name, repo_type="dataset")
                if Path(cached).resolve() != archive_path.resolve():
                    archive_path.write_bytes(Path(cached).read_bytes())
            except Exception as exc:
                errors.append(f"hf_hub_download: {exc}")

        if not archive_path.exists():
            sources = ", ".join(errors) if errors else "no sources configured"
            raise RuntimeError(f"Could not fetch {archive_name}. Tried: {sources}.")

    print(f"Extracting {archive_path.name} to {data_path} ...")
    if archive_name.endswith((".tar.gz", ".tgz")):
        import tarfile

        with tarfile.open(archive_path) as tar:
            tar.extractall(data_path)
    elif archive_name.endswith(".zip"):
        import zipfile

        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(data_path)
    else:
        raise ValueError(f"Unsupported archive format: {archive_name}")

    assert dataset_dir.exists(), (
        f"Extraction of {archive_path.name} did not produce {dataset_dir}."
    )
    print(f"Ready: {dataset_dir}")
    return dataset_dir


# Default: Serengeti balanced dataset (same as Exercises 1 and 2).
DATASET_PATH = ensure_dataset(
    DATA_PATH,
    archive_name="ser_balanced.tar.gz",
    extracted_subdir="ser/ser_balanced",
    gdrive_id="1iRlZue4-Udg_lA9RCYtg3zhctqbbPhX7",
    hf_repo="marco-willi/ser_balanced",
)

# --- Alternative: CCT-20 (8 balanced classes) or Kgalagadi (6, imbalanced) ---
# Uncomment one of the blocks below to use a different camera-trap dataset.
#
# DATASET_PATH = ensure_dataset(
#     DATA_PATH,
#     archive_name="cct20.tar.gz",
#     extracted_subdir="cct20",
#     gdrive_id="105DkEQcFhgWsQEzKh6p-u2QMMuUc2yt2",
#     hf_repo="marco-willi/camera-trap-cct20",
# )
#
# DATASET_PATH = ensure_dataset(
#     DATA_PATH,
#     archive_name="kgalagadi.tar.gz",
#     extracted_subdir="kgalagadi",
#     gdrive_id="129vX_GF4vUgwRlyLpx5BNPf6lI2n89Wp",
# )

print(f"Dataset path: {DATASET_PATH}")

In [ ]:
# Quick check: list classes and count
raw_train = ImageFolder(root=DATASET_PATH / "train")
raw_val = ImageFolder(root=DATASET_PATH / "val")
NUM_CLASSES = len(raw_train.classes)
class_names = raw_train.classes

print(f"Classes ({NUM_CLASSES}): {class_names}")
print(f"Training images: {len(raw_train)}")
print(f"Validation images: {len(raw_val)}")

In [ ]:
# Transforms: same ImageNet preprocessing as previous exercises
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

eval_transforms = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]
)

train_transforms = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
    ]
)

# For feature extraction we use deterministic eval transforms
train_dataset_eval = ImageFolder(root=DATASET_PATH / "train", transform=eval_transforms)
val_dataset = ImageFolder(root=DATASET_PATH / "val", transform=eval_transforms)

# For fine-tuning we use augmented train transforms
train_dataset_aug = ImageFolder(root=DATASET_PATH / "train", transform=train_transforms)

BATCH_SIZE = 32

train_loader_eval = DataLoader(
    train_dataset_eval, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
train_loader_aug = DataLoader(
    train_dataset_aug, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)

print(f"Train batches: {len(train_loader_eval)}, Val batches: {len(val_loader)}")

## Part 1 — Extract Features

In Exercise 2, we fine-tuned a ResNet-50 end-to-end. But do we always need to train? A pretrained model already knows a lot about images. Let's extract its representations and see how far they take us *without any training at all*.

In [ ]:
# Load a pretrained ConvNeXt in feature-extraction mode (num_classes=0 removes the head)
encoder = timm.create_model(
    "convnext_tiny.dinov3_lvd1689m", pretrained=True, num_classes=0
)
encoder = encoder.to(device)
encoder.eval()

# Check the embedding dimension
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224, device=device)
    dummy_out = encoder(dummy)
    EMBED_DIM = dummy_out.shape[1]

print("Encoder: convnext_tiny.dinov3_lvd1689m")
print(f"Embedding dimension: {EMBED_DIM}")
print(f"Total parameters: {sum(p.numel() for p in encoder.parameters()):,}")

Extract embeddings for **all** training and validation images. Loop over each DataLoader, pass images through the encoder with `torch.no_grad()`, and collect the outputs.

Store the results as NumPy arrays: `train_features`, `train_labels`, `val_features`, `val_labels`.

In [ ]:
# YOUR CODE HERE
def extract_features(encoder, data_loader, device="cpu"):
    """Extract features from a pretrained encoder for all images in a DataLoader."""
    all_features = []
    all_labels = []
    encoder.eval()
    with torch.no_grad():
        for images, labels in tqdm(data_loader, desc="Extracting features"):
            images = images.to(device)
            features = encoder(images)
            all_features.append(features.cpu())
            all_labels.append(labels)
    return torch.cat(all_features).numpy(), torch.cat(all_labels).numpy()


t0 = time.time()
train_features, train_labels = extract_features(encoder, train_loader_eval, device)
val_features, val_labels = extract_features(encoder, val_loader, device)
extraction_time = time.time() - t0

print(f"\nTrain features: {train_features.shape}")
print(f"Val features:   {val_features.shape}")
print(f"Extraction time: {extraction_time:.1f}s")

In [ ]:
assert train_features.shape == (len(train_dataset_eval), EMBED_DIM), (
    f"Expected ({len(train_dataset_eval)}, {EMBED_DIM}), got {train_features.shape}"
)
assert val_features.shape == (len(val_dataset), EMBED_DIM), (
    f"Expected ({len(val_dataset)}, {EMBED_DIM}), got {val_features.shape}"
)
assert train_labels.shape == (len(train_dataset_eval),)
assert val_labels.shape == (len(val_dataset),)
print("OK")

The features are now fixed vectors — we won't touch the encoder again until Part 4. Everything from here runs on these cached embeddings, which makes experimentation *fast*.

## Part 2 — k-NN Baseline (No Training)

The simplest way to classify with embeddings: find the nearest neighbours and vote. No parameters to learn, no loss function, no optimizer. Just geometry.

**Question:** How well do you think a k-NN classifier will work on these features? Remember, the model was trained with DINOv3 self-supervised learning on a large image dataset (LVD-1689M), not on camera trap images.

<details>
<summary>Click to reveal answer</summary>

Surprisingly well! Models trained with DINOv3 learn very general visual features — edges, textures, shapes, object parts — that transfer across domains. The embedding space already clusters similar-looking images together, even for categories the model has never been explicitly trained on.

</details>

Fit a `KNeighborsClassifier` with `n_neighbors=5` on the training features and predict on the validation features. Print the accuracy and `classification_report`.

In [ ]:
# YOUR CODE HERE
t0 = time.time()

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train_features, train_labels)

knn_preds = knn.predict(val_features)
knn_time = time.time() - t0

knn_acc = (knn_preds == val_labels).mean()
print(f"k-NN accuracy: {knn_acc:.3f}")
print(f"k-NN fit + predict time: {knn_time:.2f}s")
print()
print(classification_report(val_labels, knn_preds, target_names=class_names))

In [ ]:
assert 0.0 < knn_acc <= 1.0, f"Accuracy should be between 0 and 1, got {knn_acc}"
assert len(knn_preds) == len(val_labels)
print(f"OK — k-NN accuracy: {knn_acc:.1%}")

**Question:** What is the advantage of k-NN as a first check?

<details>
<summary>Click to reveal answer</summary>

k-NN has **zero trainable parameters** and takes seconds to run. It gives you an immediate sanity check: *are these features any good at all?* If k-NN already gets reasonable accuracy, the features capture meaningful structure and it's worth investing in more sophisticated classifiers. If k-NN fails completely, you know the features don't separate your classes and might need a different backbone or more data.

</details>

## Part 3 — Linear Probe

Can a single linear layer do better than k-NN? A linear probe learns a weight matrix that maps embeddings to class logits — still very lightweight, but it optimizes for classification directly.

Define a single `nn.Linear(EMBED_DIM, NUM_CLASSES)` layer. Train it for 20 epochs on the cached training features using `nn.CrossEntropyLoss` and `Adam` with `lr=1e-3`.

**Important:** We train on the *extracted features*, not on images. This is fast because we skip the expensive encoder forward pass.

<details>
<summary>Hint: training on features</summary>

Convert features and labels to tensors, create a `TensorDataset` and `DataLoader`, then write a standard training loop:

```python
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(torch.from_numpy(train_features), torch.from_numpy(train_labels).long())
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
```

</details>

In [ ]:
# YOUR CODE HERE
from torch.utils.data import TensorDataset

# Wrap features in TensorDatasets
train_feat_ds = TensorDataset(
    torch.from_numpy(train_features),
    torch.from_numpy(train_labels).long(),
)
val_feat_ds = TensorDataset(
    torch.from_numpy(val_features),
    torch.from_numpy(val_labels).long(),
)
train_feat_dl = DataLoader(train_feat_ds, batch_size=64, shuffle=True)
val_feat_dl = DataLoader(val_feat_ds, batch_size=64, shuffle=False)

# Define the linear probe
linear_probe = nn.Linear(EMBED_DIM, NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(linear_probe.parameters(), lr=1e-3)

NUM_EPOCHS_LP = 20
lp_history = {"train_loss": [], "train_acc": [], "val_acc": []}

t0 = time.time()

for epoch in range(NUM_EPOCHS_LP):
    # Train
    _ = linear_probe.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for feats, labels in train_feat_dl:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = linear_probe(feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * feats.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += feats.size(0)

    lp_history["train_loss"].append(running_loss / total)
    lp_history["train_acc"].append(correct / total)

    # Evaluate
    _ = linear_probe.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for feats, labels in val_feat_dl:
            feats, labels = feats.to(device), labels.to(device)
            logits = linear_probe(feats)
            correct += (logits.argmax(1) == labels).sum().item()
            total += feats.size(0)
    lp_history["val_acc"].append(correct / total)

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch + 1:2d}/{NUM_EPOCHS_LP} — "
            f"train loss: {lp_history['train_loss'][-1]:.4f}, "
            f"train acc: {lp_history['train_acc'][-1]:.3f}, "
            f"val acc: {lp_history['val_acc'][-1]:.3f}"
        )

lp_time = time.time() - t0
lp_acc = lp_history["val_acc"][-1]
print(f"\nLinear probe val accuracy: {lp_acc:.3f}")
print(f"Training time: {lp_time:.1f}s")

In [ ]:
assert 0.0 < lp_acc <= 1.0, f"Accuracy should be between 0 and 1, got {lp_acc}"
assert len(lp_history["train_loss"]) == NUM_EPOCHS_LP
print(f"OK — Linear probe accuracy: {lp_acc:.1%}")

Let's visualize the linear probe training.

In [ ]:
epochs = range(1, NUM_EPOCHS_LP + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
_ = axes[0].plot(epochs, lp_history["train_loss"], "o-", markersize=3)
_ = axes[0].set_xlabel("Epoch")
_ = axes[0].set_ylabel("Loss")
_ = axes[0].set_title("Linear Probe — Training Loss")
_ = axes[0].grid(True, alpha=0.3)

# Accuracy
_ = axes[1].plot(epochs, lp_history["train_acc"], "o-", markersize=3, label="Train")
_ = axes[1].plot(epochs, lp_history["val_acc"], "o-", markersize=3, label="Validation")
_ = axes[1].set_xlabel("Epoch")
_ = axes[1].set_ylabel("Accuracy")
_ = axes[1].set_title("Linear Probe — Accuracy")
_ = axes[1].legend()
_ = axes[1].grid(True, alpha=0.3)

_ = plt.tight_layout()
plt.show()

**Question:** How does the linear probe compare to k-NN? Why might a learned linear layer outperform nearest neighbours?

<details>
<summary>Click to reveal answer</summary>

The linear probe typically matches or slightly beats k-NN. It can outperform k-NN because:
1. It **optimizes a decision boundary** directly for the classification task (cross-entropy loss), rather than relying on local distances.
2. It can handle **class imbalance** better through the loss function.
3. It's less sensitive to the **curse of dimensionality** in high-dimensional spaces where distance-based methods can struggle.

Both methods are fast because they operate on cached features, not raw images.

</details>

## Part 4 — Partial Fine-Tuning

k-NN and the linear probe treat the encoder features as fixed. What if we let the model *adapt* its representations to our specific task? We unfreeze a few layers and train end-to-end — but with a much lower learning rate so we don't destroy the pretrained knowledge.

Load a fresh `convnext_tiny.dinov3_lvd1689m` with a classification head for our classes. Freeze all layers, then unfreeze the **last ConvNeXt stage** and the **classification head**.

<details>
<summary>Hint: finding the right layers</summary>

In `timm` ConvNeXts, the convolutional stages live in `model.stages`. The last stage is `model.stages[-1]`. The classification head (which includes the final norm) is `model.head`.

```python
# Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last stage + head
for param in model.stages[-1].parameters():
    param.requires_grad = True
for param in model.head.parameters():
    param.requires_grad = True
```

</details>

In [ ]:
# YOUR CODE HERE
ft_model = timm.create_model(
    "convnext_tiny.dinov3_lvd1689m", pretrained=True, num_classes=NUM_CLASSES
)
ft_model = ft_model.to(device)

# Freeze everything
for param in ft_model.parameters():
    param.requires_grad = False

# Unfreeze last ConvNeXt stage + classification head
for param in ft_model.stages[-1].parameters():
    param.requires_grad = True
for param in ft_model.head.parameters():
    param.requires_grad = True

trainable_params = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in ft_model.parameters())
print(
    f"Trainable parameters: {trainable_params:,} / {total_params:,} ({trainable_params / total_params:.1%})"
)

In [ ]:
assert trainable_params < total_params, "Not all params should be trainable"
assert trainable_params > NUM_CLASSES, "Should have more than just the head"
print(f"OK — {trainable_params:,} trainable params")

Now train the partially unfrozen model end-to-end for 5 epochs. Use `nn.CrossEntropyLoss` and `Adam` with `lr=1e-4` (10x lower than the linear probe — we want gentle updates).

**Note:** This is slower than the linear probe because we're now passing images through the full model on every batch.

In [ ]:
def train_one_epoch(model, data_loader, criterion, optimizer, device="cpu"):
    """Train for one epoch. Returns (avg_loss, accuracy)."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(data_loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, data_loader, criterion, device="cpu"):
    """Evaluate on a dataset. Returns (avg_loss, accuracy)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(data_loader, desc="Val", leave=False):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)

    return running_loss / total, correct / total

Write the training loop: 5 epochs, store metrics in a `ft_history` dict with keys `"train_loss"`, `"train_acc"`, `"val_loss"`, `"val_acc"`.

In [ ]:
# YOUR CODE HERE
ft_criterion = nn.CrossEntropyLoss()
ft_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, ft_model.parameters()), lr=1e-4
)

NUM_EPOCHS_FT = 5
ft_history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

t0 = time.time()

for epoch in range(NUM_EPOCHS_FT):
    epoch_t0 = time.time()

    train_loss, train_acc = train_one_epoch(
        ft_model, train_loader_aug, ft_criterion, ft_optimizer, device
    )
    val_loss, val_acc = evaluate(ft_model, val_loader, ft_criterion, device)

    ft_history["train_loss"].append(train_loss)
    ft_history["train_acc"].append(train_acc)
    ft_history["val_loss"].append(val_loss)
    ft_history["val_acc"].append(val_acc)

    elapsed = time.time() - epoch_t0
    print(
        f"Epoch {epoch + 1}/{NUM_EPOCHS_FT} ({elapsed:.1f}s) — "
        f"train loss: {train_loss:.4f}, train acc: {train_acc:.3f} | "
        f"val loss: {val_loss:.4f}, val acc: {val_acc:.3f}"
    )

ft_time = time.time() - t0
ft_acc = ft_history["val_acc"][-1]
print(f"\nFine-tuning time: {ft_time:.1f}s")

In [ ]:
assert 0.0 < ft_acc <= 1.0, f"Accuracy should be between 0 and 1, got {ft_acc}"
assert len(ft_history["train_loss"]) == NUM_EPOCHS_FT
print(f"OK — Fine-tuning accuracy: {ft_acc:.1%}")

In [ ]:
epochs = range(1, NUM_EPOCHS_FT + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
_ = axes[0].plot(epochs, ft_history["train_loss"], "o-", label="Train")
_ = axes[0].plot(epochs, ft_history["val_loss"], "o-", label="Validation")
_ = axes[0].set_xlabel("Epoch")
_ = axes[0].set_ylabel("Loss")
_ = axes[0].set_title("Partial Fine-Tuning — Loss")
_ = axes[0].legend()
_ = axes[0].grid(True, alpha=0.3)

# Accuracy
_ = axes[1].plot(epochs, ft_history["train_acc"], "o-", label="Train")
_ = axes[1].plot(epochs, ft_history["val_acc"], "o-", label="Validation")
_ = axes[1].set_xlabel("Epoch")
_ = axes[1].set_ylabel("Accuracy")
_ = axes[1].set_title("Partial Fine-Tuning — Accuracy")
_ = axes[1].legend()
_ = axes[1].grid(True, alpha=0.3)

_ = plt.tight_layout()
plt.show()

**Question:** Did partial fine-tuning improve over the linear probe? When might it *not* help?

<details>
<summary>Click to reveal answer</summary>

Partial fine-tuning typically improves accuracy because the last convolutional stage can adapt its feature maps to the target domain (e.g., focusing on animal shapes rather than ImageNet-typical features).

It might **not** help (or even hurt) when:
1. **The dataset is very small** — unfreezing more parameters increases overfitting risk.
2. **The domain is very similar to ImageNet** — the pretrained features are already optimal.
3. **The learning rate is too high** — aggressive updates destroy pretrained knowledge ("catastrophic forgetting").

This is the core trade-off: more trainable parameters = more capacity to adapt, but also more risk of overfitting on small datasets.

</details>

## Part 5 — Summary Table

Let's compare all three strategies side by side.

Fill in the summary table with your results. Compute the number of trainable parameters for each strategy and format the table.

In [ ]:
# YOUR CODE HERE
lp_params = EMBED_DIM * NUM_CLASSES + NUM_CLASSES  # weight + bias

results = [
    ("k-NN (k=5)", 0, knn_acc, knn_time),
    ("Linear probe", lp_params, lp_acc, lp_time),
    ("Partial fine-tune", trainable_params, ft_acc, ft_time),
]

print(f"{'Strategy':<22} {'Trainable params':>17} {'Val accuracy':>13} {'Time':>10}")
print("-" * 66)
for name, params, acc, t in results:
    params_str = f"{params:,}" if params > 0 else "0"
    print(f"{name:<22} {params_str:>17} {acc:>12.1%} {t:>9.1f}s")

In [ ]:
assert len(results) == 3, "Should have 3 strategies"
print("OK")

Let's also visualize the comparison.

In [ ]:
strategy_names = [r[0] for r in results]
accuracies = [r[2] for r in results]
times = [r[3] for r in results]
colors = ["#4C72B0", "#55A868", "#C44E52"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
bars = axes[0].bar(strategy_names, accuracies, color=colors)
for bar, acc in zip(bars, accuracies, strict=False):
    _ = axes[0].text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 0.005,
        f"{acc:.1%}",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
_ = axes[0].set_ylabel("Validation Accuracy")
_ = axes[0].set_title("Accuracy Comparison")
_ = axes[0].set_ylim(0, 1.05)
_ = axes[0].grid(True, alpha=0.3, axis="y")

# Time comparison
bars = axes[1].bar(strategy_names, times, color=colors)
for bar, t in zip(bars, times, strict=False):
    _ = axes[1].text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 0.5,
        f"{t:.1f}s",
        ha="center",
        va="bottom",
        fontweight="bold",
    )
_ = axes[1].set_ylabel("Time (seconds)")
_ = axes[1].set_title("Training Time Comparison")
_ = axes[1].grid(True, alpha=0.3, axis="y")

_ = plt.suptitle("Adaptation Strategy Comparison", fontsize=14)
_ = plt.tight_layout()
plt.show()

## Part 6 — When to Use What?

Answer the following questions in your own words before revealing the answers.

**Question 1:** You have 50 labeled images and need a classifier by tomorrow. Which strategy do you pick?

<details>
<summary>Click to reveal answer</summary>

**k-NN** or **linear probe**. With only 50 images, fine-tuning risks overfitting. k-NN needs zero training; a linear probe trains in seconds. Both leverage the pretrained features without modifying them, which is safest with very little data.

</details>

**Question 2:** You have 100,000 labeled images and a GPU cluster. Which strategy gives the best accuracy?

<details>
<summary>Click to reveal answer</summary>

**Full fine-tuning** (unfreeze all layers). With enough data, the model can safely adapt all its representations to the target domain. This gives the best accuracy because every layer can specialize. The risk of overfitting is low when you have 100k+ images.

</details>

**Question 3:** Your target domain is very different from ImageNet (e.g., satellite imagery, medical scans). Does this change your strategy?

<details>
<summary>Click to reveal answer</summary>

Yes! When the domain shift is large, the pretrained features are less directly useful. A linear probe may underperform because the frozen features don't capture domain-specific patterns. In this case, **fine-tuning more layers** helps the model learn new low-level features (textures, colors) that differ from natural images. However, you still benefit from the pretrained *architecture* and *learned optimization landscape*, so starting from pretrained weights is almost always better than training from scratch.

</details>

## Summary

In this exercise you compared three ways to reuse a pretrained model:

1. **k-NN** — zero training, instant results, great for sanity checks
2. **Linear probe** — trains only a single layer on cached features, fast and effective
3. **Partial fine-tuning** — adapts the last convolutional stage, best accuracy but slower and riskier on small data

**Key takeaways:**
- Pretrained models are powerful feature extractors — even without training, k-NN on pretrained features works surprisingly well
- More trainable parameters ≠ better: it depends on dataset size and domain gap
- Always start simple (k-NN / linear probe) before investing in fine-tuning
- The feature extraction step is the bottleneck — cache embeddings whenever possible
